In [0]:
%run ./world_export_config

In [0]:
#Read Bronze layer

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

storage_account = "worldexportsdata"
CONTAINER = "world-exports"
bronze_path = f"abfss://{CONTAINER}@{storage_account}.dfs.core.windows.net/bronze/world_exports"
silver_path = f"abfss://{CONTAINER}@{storage_account}.dfs.core.windows.net/silver/world_exports_cleaned"

bronze_df = spark.read \
    .format("delta") \
    .load(bronze_path)

print(f"Bronze records: {bronze_df.count()}")
bronze_df.printSchema()

In [0]:
# Data quality report before cleaning

print("=== SILVER LAYER: DATA QUALITY REPORT ===\n")

total = bronze_df.count()
print(f"Total records:          {total}")

# Count nulls per column
null_counts = bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in bronze_df.columns
])
print("\nNull counts per column:")
display(null_counts)

# Duplicate count
dupes = total - bronze_df.dropDuplicates().count()
print(f"\nDuplicate rows:         {dupes}")

# Negative export values
neg_exports = bronze_df.filter(F.col("export_value_usd") < 0).count()
print(f"Negative export values: {neg_exports}")

# Future year records
future_years = bronze_df.filter(F.col("year") > 2024).count()
print(f"Future year records:    {future_years}")

# Bad quantity format (contains "units")
bad_qty = bronze_df.filter(F.col("quantity").cast("string").contains("units")).count()
print(f"Bad quantity format:    {bad_qty}")

In [0]:
# Apply all cleaning transformations

from pyspark.sql.functions import regexp_replace, trim, initcap, abs as spark_abs, when, col, median

silver_df = bronze_df

# 1. Remove duplicates
silver_df = silver_df.dropDuplicates()

# 2. Fix quantity string errors
# We use regexp_replace to remove ' units' and cast to double
silver_df = silver_df.withColumn('quantity', 
                   F.regexp_replace(F.col('quantity').cast('string'), ' units', '')
                   .cast('double'))

# 3. Fix negative export values
silver_df = silver_df.withColumn('export_value_usd', F.abs(F.col('export_value_usd')))

# 4. Fill missing numeric values with median
# Spark doesn't have a built-in .median() in fillna, so we calculate it first
for col_name in ['export_value_usd', 'quantity', 'growth_rate_pct']:
    median_val = silver_df.approxQuantile(col_name, [0.5], 0.01)[0]
    silver_df = silver_df.na.fill({col_name: median_val})

# 5. Fill missing categorical values
# Get the mode for product_category
mode_row = silver_df.groupby('product_category').count().orderBy('count', ascending=False).first()
mode_val = mode_row['product_category'] if mode_row else 'Unknown'

silver_df = silver_df.na.fill({'country': 'Unknown', 'product_category': mode_val})

# 6. Standardize text
silver_df = silver_df.withColumn('country', F.initcap(F.trim(F.col('country'))))
silver_df = silver_df.withColumn('product_category', F.trim(F.col('product_category')))

# Output results
print(f"Cleaned records: {silver_df.count()}")
# To check nulls in Spark, we aggregate the count of nulls per column
null_counts = silver_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_df.columns])
null_counts.show()

In [0]:
# Save Silver Delta table

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Silver layer saved to ADLS Gen2!")
display(silver_df.limit(10))